In [1]:
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, LinearSegmentedColormap
from matplotlib.ticker import LogFormatterMathtext
import matplotlib.ticker as ticker
from scipy.ndimage import gaussian_filter
import numpy as np
import pandas as pd
import dask
import zarr
import xarray as xr
import os

In [2]:
# ============== LOAD LLC ==============
llc_path = '/orcd/data/abodner/002/cody/LLC_patch/LLC4320_face1_i2880-3600_j720-1440.zarr'
llc_patch_full = xr.open_dataset(llc_path, consolidated=True)

# ============== LOAD EMULATORS ==============
emulator_configs = [
    {
        'name': 'exp3_ckpt38',
        'key': 'emulator_1',
        'path': '/orcd/data/abodner/002/cody/inference_patch/2026-05-29-eval:Samudra_LLC:long_curriculum_strides=3_ckpt-38-7days-14748021/predictions_4d.zarr',
        'desc': ''   
    },
    {
        'name': 'A_ckpt38',
        'key': 'emulator_2',
        'path': '/orcd/data/abodner/002/cody/inference_patch/2026-06-08-eval:Samudra_LLC:A_ckpt-38-15626896/predictions_4d.zarr',
        'desc': ''
    },
        {
        'name': 'B_ckpt38',
        'key': 'emulator_3',
        'path': '/orcd/data/abodner/002/cody/inference_patch/2026-06-08-eval:Samudra_LLC:B_ckpt-38-15632827/predictions_4d.zarr',
        'desc': ''
    },
]

# ============== OPEN EMULATOR DATASETS ==============
emulator_patches_raw = {}
for cfg in emulator_configs:
    emulator_patches_raw[cfg['key']] = xr.open_dataset(cfg['path'], consolidated=True)
    print(f"Loaded {cfg['name']}: {cfg['desc']}")

# ============== TIME MATCHING ==============
def normalize_times(times):
    return pd.DatetimeIndex([
        pd.Timestamp(
            int(t.year), int(t.month), int(t.day),
            int(t.hour), int(t.minute), int(t.second)
        )
        if hasattr(t, "year")
        else pd.Timestamp(t).floor("s")
        for t in times
    ])

llc_times_norm = normalize_times(llc_patch_full.time.values)

common_times = llc_times_norm
for cfg in emulator_configs:
    emulator_times_norm = normalize_times(emulator_patches_raw[cfg['key']].time.values)
    common_times = common_times.intersection(emulator_times_norm)

common_times = common_times.sort_values()

llc_mask = llc_times_norm.isin(common_times)
llc_patch = llc_patch_full.isel(time=llc_mask)

print(f"LLC subset to {len(common_times)} common times")

# ============== PORT GRID VARS & BUILD UNIFIED STRUCTURE ==============
grid_vars = ['XC', 'YC', 'rA', 'Z']

emulator_patches = {}
for cfg in emulator_configs:
    patch_raw = emulator_patches_raw[cfg['key']]
    patch_times_norm = normalize_times(patch_raw.time.values)

    patch_mask = patch_times_norm.isin(common_times)
    patch = patch_raw.isel(time=patch_mask)

    for gv in grid_vars:
        patch[gv] = llc_patch[gv]

    emulator_patches[cfg['key']] = patch

# ============== UNIFIED REFERENCE LISTS ==============
emulator_info = [(cfg['name'], cfg['key']) for cfg in emulator_configs]
n_emulators = len(emulator_info)

all_patches = {'llc': llc_patch}
all_patches.update(emulator_patches)

print(f"\n=== Setup complete: LLC + {n_emulators} emulators ===")
for name, key in emulator_info:
    print(f"  {name} ({key})")

Loaded exp3_ckpt38: 
Loaded A_ckpt38: 
Loaded B_ckpt38: 
LLC subset to 40 common times

=== Setup complete: LLC + 3 emulators ===
  exp3_ckpt38 (emulator_1)
  A_ckpt38 (emulator_2)
  B_ckpt38 (emulator_3)


In [3]:
selected_time_range = [0, 24]   # inclusive indices
stepping = 1                    # 1 = every timestep, 4 = every 4th timestep

start_idx, end_idx = selected_time_range

# ----------------------------------------
# First subset LLC
# ----------------------------------------
llc_patch = llc_patch.isel(
    time=slice(start_idx, end_idx + 1, stepping)
)

# ----------------------------------------
# Then subset each emulator safely
# Handles shorter emulator runs automatically
# ----------------------------------------
emulator_patches_subset = {}

for key, patch in emulator_patches.items():

    max_time = patch.sizes['time']

    # Prevent indexing past emulator length
    safe_end_idx = min(end_idx, max_time - 1)

    patch_subset = patch.isel(
        time=slice(start_idx, safe_end_idx + 1, stepping)
    )

    emulator_patches_subset[key] = patch_subset

emulator_patches = emulator_patches_subset

# ----------------------------------------
# Match LLC length to shortest emulator
# ----------------------------------------
min_time_len = min(
    [llc_patch.sizes['time']] +
    [patch.sizes['time'] for patch in emulator_patches.values()]
)

llc_patch = llc_patch.isel(time=slice(0, min_time_len))

emulator_patches = {
    key: patch.isel(time=slice(0, min_time_len))
    for key, patch in emulator_patches.items()
}

# ----------------------------------------
# Rebuild combined dict
# ----------------------------------------
all_patches = {'llc': llc_patch}
all_patches.update(emulator_patches)

# ----------------------------------------
# Diagnostics
# ----------------------------------------
print(f"Subset to time indices {start_idx}:{end_idx}")
print(f"Stepping = {stepping}")
print(f"Final synchronized length = {min_time_len}")

print(f"LLC now has {llc_patch.sizes['time']} times")

for name, key in emulator_info:
    print(
        f"{name} ({key}) now has "
        f"{emulator_patches[key].sizes['time']} times"
    )

Subset to time indices 0:24
Stepping = 1
Final synchronized length = 25
LLC now has 25 times
exp3_ckpt38 (emulator_1) now has 25 times
A_ckpt38 (emulator_2) now has 25 times
B_ckpt38 (emulator_3) now has 25 times


In [4]:
def format_time(t_val):
    """Format a time value to DD/MM/YYYY:HH regardless of cftime or datetime64."""
    try:
        # cftime objects
        return f"{t_val.day:02d}/{t_val.month:02d}/{t_val.year}:{t_val.hour:02d}h"
    except AttributeError:
        # numpy datetime64
        t_pd = pd.Timestamp(t_val)
        return f"{t_pd.day:02d}/{t_pd.month:02d}/{t_pd.year}:{t_pd.hour:02d}h"


In [5]:
omega = 7.2921e-5

for patch_name, patch in all_patches.items():
    print(f"Computing vorticity for {patch_name}...")
    
    f_0 = np.abs(2 * omega * np.sin(np.deg2rad(patch['YC'].values)))
    
    dx = np.sqrt(patch['rA'].values)
    dy = dx.copy()
    
    U = patch['U'].values
    V = patch['V'].values
    
    dvdx = (np.roll(V, -1, axis=3) - np.roll(V, 1, axis=3)) / (2 * dx[np.newaxis, np.newaxis, :, :])
    dudy = (np.roll(U, -1, axis=2) - np.roll(U, 1, axis=2)) / (2 * dy[np.newaxis, np.newaxis, :, :])
    
    vort = dvdx - dudy
    vort_normalized = vort / f_0[np.newaxis, np.newaxis, :, :]
    
    patch['vorticity'] = (('time', 'k', 'j', 'i'), vort_normalized)
    print(f"  ✓ vorticity: {vort_normalized.shape}")

print("Done computing vorticity!")

Computing vorticity for llc...


  ✓ vorticity: (25, 51, 720, 720)
Computing vorticity for emulator_1...
  ✓ vorticity: (25, 51, 720, 720)
Computing vorticity for emulator_2...
  ✓ vorticity: (25, 51, 720, 720)
Computing vorticity for emulator_3...
  ✓ vorticity: (25, 51, 720, 720)
Done computing vorticity!


In [ ]:
omega = 7.2921e-5

for patch_name, patch in all_patches.items():
    print(f"Computing strain for {patch_name}...")
    
    f_0 = np.abs(2 * omega * np.sin(np.deg2rad(patch['YC'].values)))
    
    dx = np.sqrt(patch['rA'].values)
    dy = dx.copy()
    
    U = patch['U'].values
    V = patch['V'].values
    
    u_x = (np.roll(U, -1, axis=3) - np.roll(U, 1, axis=3)) / (2 * dx[np.newaxis, np.newaxis, :, :])
    u_y = (np.roll(U, -1, axis=2) - np.roll(U, 1, axis=2)) / (2 * dy[np.newaxis, np.newaxis, :, :])
    v_x = (np.roll(V, -1, axis=3) - np.roll(V, 1, axis=3)) / (2 * dx[np.newaxis, np.newaxis, :, :])
    v_y = (np.roll(V, -1, axis=2) - np.roll(V, 1, axis=2)) / (2 * dy[np.newaxis, np.newaxis, :, :])
    
    sigma_n = u_x - v_y
    sigma_s = v_x + u_y
    sigma = np.sqrt(sigma_n**2 + sigma_s**2)
    
    sigma_normalized = sigma / f_0[np.newaxis, np.newaxis, :, :]
    
    patch['strain'] = (('time', 'k', 'j', 'i'), sigma_normalized)
    print(f"  ✓ strain: {sigma_normalized.shape}")

print("Done computing strain!")

Computing strain for llc...


In [ ]:
mixing_vars = ['vorticity', 'strain']
colormaps = {'vorticity': 'magma', 'strain': 'plasma'}

ref_patch = emulator_patches[emulator_info[0][1]]

for var in mixing_vars:
    print(f"Generating plots for {var}...")
    
    os.makedirs(f'figs/mixing/{var}', exist_ok=True)
    
    n_times = len(ref_patch.time)
    time_indices = list(range(n_times))
    nrows = len(time_indices)
    ncols_fields = 1 + n_emulators
    ncols_diff = n_emulators
    
    cmap = colormaps[var]
    
    # ==================== PLOT 1: Surface fields ====================
    fig, axes = plt.subplots(nrows, ncols_fields, figsize=(3.6*ncols_fields, 3*nrows), dpi=200)
    
    if nrows == 1 and ncols_fields > 1:
        axes = axes.reshape(1, -1)
    elif nrows > 1 and ncols_fields == 1:
        axes = axes.reshape(-1, 1)
    elif nrows == 1 and ncols_fields == 1:
        axes = axes.reshape(1, 1)
    
    for row, t in enumerate(time_indices):
        time_str = format_time(ref_patch.time.values[t])
        
        # Collect all surface fields
        fields = [llc_patch.isel(time=t, k=0)[var]]
        for emu_name, emu_key in emulator_info:
            fields.append(emulator_patches[emu_key].isel(time=t, k=0)[var])
        
        # Shared quantile-based vmin/vmax
        combined_data = np.concatenate([f.values.flatten() for f in fields])
        vmin = float(np.nanpercentile(combined_data, 0.1))
        vmax = float(np.nanpercentile(combined_data, 99.9))
        
        labels = ['LLC'] + [name for name, _ in emulator_info]
        
        for col, (field, label) in enumerate(zip(fields, labels)):
            ax = axes[row, col]
            cf = ax.contourf(field.coords.get('i', np.arange(field.shape[-1])),
                             field.coords.get('j', np.arange(field.shape[-2])), field,
                             cmap=cmap, vmin=vmin, vmax=vmax, levels=30)
            ax.set_title(f'{label} {var} {time_str}', fontsize=8)
            plt.colorbar(cf, ax=ax)
    
    plt.tight_layout()
    plt.savefig(f'figs/mixing/{var}/surface_{var}_fields.png')
    plt.close()
    
    # ==================== PLOT 2: Difference fields ====================
    fig, axes = plt.subplots(nrows, ncols_diff, figsize=(4*ncols_diff, 3*nrows), dpi=200)
    
    if nrows == 1 and ncols_diff > 1:
        axes = axes.reshape(1, -1)
    elif nrows > 1 and ncols_diff == 1:
        axes = axes.reshape(-1, 1)
    elif nrows == 1 and ncols_diff == 1:
        axes = axes.reshape(1, 1)
    
    for row, t in enumerate(time_indices):
        time_str = format_time(ref_patch.time.values[t])
        
        llc_vis = llc_patch.isel(time=t, k=0)[var]
        
        diffs = []
        for emu_name, emu_key in emulator_info:
            emu_vis = emulator_patches[emu_key].isel(time=t, k=0)[var]
            diffs.append(llc_vis.values - emu_vis.values)
        
        abs_max = np.max([np.abs(d).max() for d in diffs])
        vmin_d, vmax_d = -abs_max, abs_max
        
        row_axes = [axes[row, col] for col in range(ncols_diff)]
        
        for col, ((emu_name, _), diff) in enumerate(zip(emulator_info, diffs)):
            ax = row_axes[col]
            cf = ax.contourf(llc_vis.coords.get('i', np.arange(llc_vis.shape[-1])),
                             llc_vis.coords.get('j', np.arange(llc_vis.shape[-2])), diff,
                             cmap="bwr", vmin=vmin_d, vmax=vmax_d, levels=30)
            short_name = emu_name.replace('Emulator ', 'Em')
            ax.set_title(f'LLC - {short_name} {var} {time_str}', fontsize=8)
        
        fig.colorbar(cf, ax=row_axes, orientation='vertical',
                     fraction=0.046, pad=0.04)
    
    plt.savefig(f'figs/mixing/{var}/surface_{var}_differences.png')
    plt.close()
    
    print(f"✓ Saved plots for {var}")

Generating plots for vorticity...
✓ Saved plots for vorticity
Generating plots for strain...
✓ Saved plots for strain


In [ ]:
mixing_vars = ['vorticity', 'strain']
ref_lines_mixing = {
    'vorticity': [0.5, 1.0],
    'strain': [0.5, 1.0]
}

ref_patch = emulator_patches[emulator_info[0][1]]

for var in mixing_vars:
    print(f"Generating depth error plots for {var}...")
    os.makedirs(f'figs/mixing/{var}', exist_ok=True)
    
    n_depths = llc_patch.sizes['k']
    n_times = len(ref_patch.time)
    time_indices = list(range(n_times))
    
    nrows = len(time_indices)
    ncols = n_emulators
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3*nrows), dpi=150)
    
    if nrows == 1 and ncols > 1:
        axes = axes.reshape(1, -1)
    elif nrows > 1 and ncols == 1:
        axes = axes.reshape(-1, 1)
    elif nrows == 1 and ncols == 1:
        axes = axes.reshape(1, 1)
    
    depths = np.arange(n_depths)
    
    for row, t in enumerate(time_indices):
        time_str = format_time(ref_patch.time.values[t])
        
        llc_data = llc_patch.isel(time=t)[var].values
        
        # First pass: compute all errors for shared xlim
        row_mean_errors = []
        row_median_errors = []
        
        for emu_name, emu_key in emulator_info:
            emu_data = emulator_patches[emu_key].isel(time=t)[var].values
            diff = np.abs(llc_data - emu_data)
            
            diff_flat = diff.reshape(n_depths, -1)
            mean_errors = np.nanmean(diff_flat, axis=1)
            median_errors = np.nanmedian(diff_flat, axis=1)
            
            row_mean_errors.append(mean_errors)
            row_median_errors.append(median_errors)
        
        all_errors = np.concatenate(row_mean_errors + row_median_errors)
        xmin = 0
        xmax = np.nanmax(all_errors) * 1.05
        
        # Second pass: plot
        for col, (emu_name, _) in enumerate(emulator_info):
            ax = axes[row, col]
            
            ax.scatter(row_mean_errors[col], depths, color='blue', s=30, alpha=0.7, zorder=3)
            ax.plot(row_mean_errors[col], depths, color='blue', alpha=0.4, linewidth=1.5, label='Mean')
            
            ax.scatter(row_median_errors[col], depths, color='red', s=30, alpha=0.7, zorder=3)
            ax.plot(row_median_errors[col], depths, color='red', alpha=0.4, linewidth=1.5, label='Median')
            
            for ref_val in ref_lines_mixing[var]:
                ax.axvline(x=ref_val, color='black', linestyle='--', linewidth=1.5, alpha=0.5, zorder=2)
            
            ax.set_title(f'{emu_name} {var} {time_str}', fontsize=8)
            ax.set_xlabel('Abs Error', fontsize=7)
            ax.set_ylabel('Depth (k)', fontsize=7)
            ax.set_ylim(n_depths - 1, 0)
            ax.set_xlim(xmin, xmax)
            ax.grid(alpha=0.2)
            ax.tick_params(labelsize=6)
            
            if col == 0:
                ax.legend(fontsize=6, loc='lower right')
    
    plt.tight_layout()
    plt.savefig(f'figs/mixing/{var}/depth_error_by_time.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Saved depth error plots for {var}")

print("Done!")

Generating depth error plots for vorticity...
✓ Saved depth error plots for vorticity
Generating depth error plots for strain...
✓ Saved depth error plots for strain
Done!


In [ ]:
for patch_name, patch in all_patches.items():
    print(f"Computing grad_T for {patch_name}...")
    
    Theta = patch['Theta'].values
    
    dx = np.sqrt(patch['rA'].values)
    dy = dx.copy()
    
    dT_di = (np.roll(Theta, -1, axis=3) - np.roll(Theta, 1, axis=3)) / (2 * dx[np.newaxis, np.newaxis, :, :])
    dT_dj = (np.roll(Theta, -1, axis=2) - np.roll(Theta, 1, axis=2)) / (2 * dy[np.newaxis, np.newaxis, :, :])
    
    grad_T = np.sqrt(dT_di**2 + dT_dj**2)
    
    patch['grad_T'] = (('time', 'k', 'j', 'i'), grad_T)
    print(f"  ✓ grad_T: {grad_T.shape}")

print("Done computing grad_T!")

Computing grad_T for llc...
  ✓ grad_T: (16, 51, 720, 720)
Computing grad_T for emulator_3...
  ✓ grad_T: (16, 51, 720, 720)
Computing grad_T for emulator_4...
  ✓ grad_T: (16, 51, 720, 720)
Done computing grad_T!


In [ ]:
print("Generating time-separated JPDFs...")

os.makedirs('figs/JPDFs', exist_ok=True)

ref_patch = emulator_patches[emulator_info[0][1]]
n_times = len(ref_patch.time)
time_indices = list(range(n_times))
nrows = len(time_indices)
ncols = 1 + n_emulators  # LLC + emulators

fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows), dpi=150)

if nrows == 1 and ncols > 1:
    axes = axes.reshape(1, -1)
elif nrows > 1 and ncols == 1:
    axes = axes.reshape(-1, 1)
elif nrows == 1 and ncols == 1:
    axes = axes.reshape(1, 1)

# Build patch_info from unified structure
patch_info = [('LLC', llc_patch)] + [(name, emulator_patches[key]) for name, key in emulator_info]

for row, t in enumerate(time_indices):
    time_str = format_time(ref_patch.time.values[t])
    print(f"  Processing time {t}: {time_str}")
    
    # First pass: compute all JPDFs for this time to get shared colorbar limits
    row_gradT_data = []
    
    for patch_name, patch in patch_info:
        vort = patch.vorticity.isel(time=t).values[:21]
        strain = patch.strain.isel(time=t).values[:21]
        grad_T_data = patch.grad_T.isel(time=t).values[:21]
        
        v = vort.flatten()
        s = strain.flatten()
        g = grad_T_data.flatten()
        
        mask = ~(np.isnan(v) | np.isnan(s) | np.isnan(g)) & (g > 0)
        v = v[mask]; s = s[mask]; g = g[mask]
        
        idx = np.random.choice(len(v), min(100000, len(v)), replace=False)
        v_sub = v[idx]; s_sub = s[idx]; g_sub = g[idx]
        
        counts, xedges, yedges = np.histogram2d(v_sub, s_sub, bins=200)
        g_sum, _, _ = np.histogram2d(v_sub, s_sub, bins=[xedges, yedges], weights=g_sub)
        
        counts_smooth = gaussian_filter(counts, sigma=0.75)
        g_sum_smooth = gaussian_filter(g_sum, sigma=0.75)
        
        g_mean = np.full_like(counts_smooth, np.nan)
        valid = counts_smooth > 0
        g_mean[valid] = g_sum_smooth[valid] / counts_smooth[valid]
        g_mean = np.where(g_mean > 0, g_mean, np.nan)
        
        row_gradT_data.append((v_sub, s_sub, g_sub, g_mean, xedges, yedges))
    
    # Shared colorbar limits for this row
    shared_vmin_g = min([np.nanpercentile(d[3], 1) for d in row_gradT_data])
    shared_vmax_g = max([np.nanpercentile(d[3], 99) for d in row_gradT_data])
    log_min_g = np.floor(np.log10(shared_vmin_g))
    log_max_g = np.ceil(np.log10(shared_vmax_g))
    levels_g = 10 ** np.arange(log_min_g, log_max_g + 0.5, 0.5)
    
    # Second pass: plot
    for col, (patch_name, _) in enumerate(patch_info):
        ax = axes[row, col]
        v_sub, s_sub, g_sub, g_mean, xedges, yedges = row_gradT_data[col]
        
        xc = (xedges[:-1] + xedges[1:]) / 2
        yc = (yedges[:-1] + yedges[1:]) / 2
        
        cf = ax.contourf(xc, yc, g_mean.T, levels=levels_g,
                         cmap='magma_r', norm=LogNorm(vmin=shared_vmin_g, vmax=shared_vmax_g))
        ax.scatter(v_sub, s_sub, s=0.1, alpha=0.05, color='k', rasterized=True)
        
        lim = max(np.abs(v_sub).max(), s_sub.max())
        ax.plot([0, lim], [0, lim], 'k--', linewidth=1, label=r'$\sigma = |\zeta|$')
        ax.plot([0, -lim], [0, lim], 'k--', linewidth=1)
        
        ax.set_xlabel(r'$\zeta / f_0$', fontsize=8)
        ax.set_ylabel(r'$\sigma / |f_0|$', fontsize=8)
        ax.set_title(f'{patch_name} {time_str}', fontsize=9)
        ax.tick_params(labelsize=7)
        
        if col == 0:
            ax.legend(fontsize=6)
        
        if col == ncols - 1:
            cbar = plt.colorbar(cf, ax=axes[row, :].tolist(),
                               label=r'$|\nabla T|$', fraction=0.046, pad=0.04)
            cbar.ax.yaxis.set_major_formatter(LogFormatterMathtext())
            cbar.ax.tick_params(labelsize=7)

plt.savefig('figs/JPDFs/JPDF_vort_strain_gradT_by_time.png', dpi=150, bbox_inches='tight')
plt.close()

print("✓ Saved time-separated JPDF figure")

Generating time-separated JPDFs...
  Processing time 0: 01/10/2012:15h
  Processing time 1: 01/10/2012:18h
  Processing time 2: 01/10/2012:21h
  Processing time 3: 02/10/2012:00h
  Processing time 4: 02/10/2012:03h
  Processing time 5: 02/10/2012:06h
  Processing time 6: 02/10/2012:09h
  Processing time 7: 02/10/2012:12h
  Processing time 8: 02/10/2012:15h
  Processing time 9: 02/10/2012:18h
  Processing time 10: 02/10/2012:21h
  Processing time 11: 03/10/2012:00h
  Processing time 12: 03/10/2012:03h
  Processing time 13: 03/10/2012:06h
  Processing time 14: 03/10/2012:09h
  Processing time 15: 03/10/2012:12h


findfont: Font family ['STIXGeneral'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXGeneral'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXGeneral'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXGeneral'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXNonUnicode'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXNonUnicode'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXNonUnicode'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXSizeOneSym'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXSizeTwoSym'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXSizeThreeSym'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXSizeFourSym'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXSizeFiveSym'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmsy10'] not

✓ Saved time-separated JPDF figure
